In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device  : {device}")
print(f"PyTorch : {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"RAM avail: {os.popen('free -h').read().split()[7]} GB")

Device  : cuda
PyTorch : 2.11.0+cu128
GPU     : NVIDIA RTX A4000
VRAM    : 16.8 GB
RAM avail: 123Gi GB


In [2]:
DATA_ROOT = os.path.expanduser(
    '~/GATA-Dep/extracted_all/extracted'
)

In [3]:
print(len(os.listdir(DATA_ROOT)))

238


In [4]:
DATA_ROOT  = os.path.expanduser('~/GATA-Dep/extracted_all/extracted')

TRAIN_CSV  = os.path.expanduser('~/GATA-Dep/train_split_Depression_AVEC2017.csv')
DEV_CSV    = os.path.expanduser('~/GATA-Dep/dev_split_Depression_AVEC2017.csv')
TEST_CSV   = os.path.expanduser('~/GATA-Dep/full_test_split.csv')

OUTPUT_DIR = "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
CACHE_DIR  = os.path.expanduser('~/gata_cache')

In [5]:
gata_save = os.path.join(OUTPUT_DIR, "gata_model.pt")

In [5]:
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
dev_df   = pd.read_csv(DEV_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(len(train_df))
print(len(dev_df))
print(len(test_df))

107
35
47


In [6]:
import pandas as pd

# ── Load official splits directly from your Drive ────────────
# Change these paths to where you saved the CSVs on Drive


train_df = pd.read_csv(TRAIN_CSV)
dev_df   = pd.read_csv(DEV_CSV)
test_df  = pd.read_csv(TEST_CSV)
test_df.columns = test_df.columns.str.strip()

def df_to_participant_info(df, id_col='Participant_ID',
                           label_col='PHQ8_Binary',
                           gender_col='Gender',
                           score_col='PHQ8_Score'):
    info = {}
    for _, row in df.iterrows():
        info[str(int(row[id_col]))] = {
            'gender':    int(row[gender_col]),
            'label':     int(row[label_col]),
            'phq_score': int(row[score_col]),
        }
    return info

train_participant_info = df_to_participant_info(train_df)
dev_participant_info   = df_to_participant_info(dev_df)
test_participant_info  = df_to_participant_info(
    test_df,
    id_col='Participant_ID',
    label_col='PHQ_Binary',
    score_col='PHQ_Score'
)

# Only keep what's actually on Drive
def filter_available(info, data_root):
    return {
        pid: v for pid, v in info.items()
        if os.path.isdir(os.path.join(data_root, pid)) or
           os.path.isdir(os.path.join(data_root, f"{pid}_P"))
    }

train_participant_info = filter_available(
    train_participant_info, DATA_ROOT)
dev_participant_info   = filter_available(
    dev_participant_info,   DATA_ROOT)
test_participant_info  = filter_available(
    test_participant_info,  DATA_ROOT)

print("TRAIN split:")
print(f"  Available : {len(train_participant_info)} / {len(train_df)}")
dep = sum(1 for v in train_participant_info.values() if v['label']==1)
print(f"  Dep:{dep}  Ctrl:{len(train_participant_info)-dep}")

print("\nDEV split:")
print(f"  Available : {len(dev_participant_info)} / {len(dev_df)}")
dep = sum(1 for v in dev_participant_info.values() if v['label']==1)
print(f"  Dep:{dep}  Ctrl:{len(dev_participant_info)-dep}")

print("\nTEST split:")
print(f"  Available : {len(test_participant_info)} / {len(test_df)}")
dep = sum(1 for v in test_participant_info.values() if v['label']==1)
print(f"  Dep:{dep}  Ctrl:{len(test_participant_info)-dep}")

# participant_info for caching = all three combined
all_participant_info = {
    **train_participant_info,
    **dev_participant_info,
    **test_participant_info,
}
print(f"\nTotal available: {len(all_participant_info)}")

TRAIN split:
  Available : 107 / 107
  Dep:30  Ctrl:77

DEV split:
  Available : 35 / 35
  Dep:12  Ctrl:23

TEST split:
  Available : 46 / 47
  Dep:14  Ctrl:32

Total available: 188


In [7]:
def load_participant_features(pid, data_root):

    # Check both "326" and "326_P" folder formats
    for folder_name in [str(pid), f"{pid}_P"]:
        folder = os.path.join(data_root, folder_name)
        if os.path.isdir(folder):
            break
    else:
        print(f"  MISSING folder for {pid}")
        return None

    files = {
        'covarep':  os.path.join(folder, f"{pid}_COVAREP.csv"),
        'formant':  os.path.join(folder, f"{pid}_FORMANT.csv"),
        'au':       os.path.join(folder, f"{pid}_CLNF_AUs.csv"),
        'landmark': os.path.join(folder, f"{pid}_CLNF_features3D.txt"),
        'gaze':     os.path.join(folder, f"{pid}_CLNF_gaze.txt"),
        'pose':     os.path.join(folder, f"{pid}_CLNF_pose.txt"),
    }

    # AU file can be .csv or .txt
    if not os.path.exists(files['au']):
        files['au'] = os.path.join(folder, f"{pid}_CLNF_AUs.txt")

    for name, path in files.items():
        if not os.path.exists(path):
            print(f"  MISSING {pid}/{name} in {folder}")
            return None

    try:
        covarep = pd.read_csv(files['covarep'], header=None)
        formant = pd.read_csv(files['formant'], header=None)
        covarep = covarep.apply(
            pd.to_numeric, errors='coerce').values.astype(np.float32)
        formant = formant.apply(
            pd.to_numeric, errors='coerce').values.astype(np.float32)
        min_len = min(len(covarep), len(formant))
        audio   = np.concatenate(
            [covarep[:min_len], formant[:min_len]], axis=1)

        def load_openface(path):
            df = pd.read_csv(path, sep=',')
            df.columns = df.columns.str.strip()
            drop = ['frame', 'timestamp', 'confidence', 'success']
            cols = [c for c in df.columns
                    if not any(k in c.lower() for k in drop)]
            return df[cols].select_dtypes(
                include=[np.number]).values.astype(np.float32)

        return {
            'audio':    audio,
            'au':       load_openface(files['au']),
            'landmark': load_openface(files['landmark']),
            'gaze':     load_openface(files['gaze']),
            'pose':     load_openface(files['pose']),
        }

    except Exception as e:
        print(f"  ERROR {pid}: {e}")
        return None

In [8]:
import numpy as np

WINDOW_SEC   = 6
AUDIO_HZ     = 100
VIDEO_HZ     = 25
AUDIO_WINDOW = WINDOW_SEC * AUDIO_HZ   # 600
VIDEO_WINDOW = WINDOW_SEC * VIDEO_HZ   # 150

def normalize(arr):
    mean = np.nanmean(arr, axis=0, keepdims=True)
    std  = np.nanstd (arr, axis=0, keepdims=True)
    std[std == 0] = 1.0
    arr = (arr - mean) / std
    return np.nan_to_num(
        arr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def make_windows(features):
    audio    = normalize(features['audio'])
    au       = normalize(features['au'])
    landmark = normalize(features['landmark'])
    gaze     = normalize(features['gaze'])
    pose     = normalize(features['pose'])

    n_wins = min(len(audio) // AUDIO_WINDOW,
                 len(au)    // VIDEO_WINDOW)
    windows = []
    for i in range(n_wins):
        a = i * AUDIO_WINDOW
        v = i * VIDEO_WINDOW
        w = {
            'audio':    audio   [a: a + AUDIO_WINDOW],
            'au':       au      [v: v + VIDEO_WINDOW],
            'landmark': landmark[v: v + VIDEO_WINDOW],
            'gaze':     gaze    [v: v + VIDEO_WINDOW],
            'pose':     pose    [v: v + VIDEO_WINDOW],
        }
        if all(w[k].shape[0] == (
                AUDIO_WINDOW if k == 'audio' else VIDEO_WINDOW)
               for k in w):
            windows.append(w)
    return windows

print(f"Window: audio={AUDIO_WINDOW} frames  video={VIDEO_WINDOW} frames")

Window: audio=600 frames  video=150 frames


In [9]:
import os
import numpy as np

def cache_all_participants(participant_info, data_root, cache_dir):
    already       = set(os.listdir(cache_dir))
    total_windows = 0
    failed        = []

    for pid, info in participant_info.items():
        fname = f"{pid}.npy"
        if fname in already:
            print(f"  {pid} — already cached")
            continue

        print(f"  {pid} ...", end="  ")
        feats = load_participant_features(pid, data_root)
        if feats is None:
            failed.append(pid); print("FAILED"); continue

        wins = make_windows(feats)
        if not wins:
            failed.append(pid); print("NO WINDOWS"); continue

        np.save(
            os.path.join(cache_dir, fname),
            {
                'audio':    np.stack([w['audio']    for w in wins]),
                'au':       np.stack([w['au']        for w in wins]),
                'landmark': np.stack([w['landmark']  for w in wins]),
                'gaze':     np.stack([w['gaze']      for w in wins]),
                'pose':     np.stack([w['pose']      for w in wins]),
                'label':    np.array([info['label']] * len(wins)),
                'gender':   np.array([info['gender']]* len(wins)),
            }
        )
        total_windows += len(wins)
        print(f"OK ({len(wins)} windows)")

    size_mb = sum(
        os.path.getsize(os.path.join(cache_dir, f))
        for f in os.listdir(cache_dir)
    ) / 1e6
    print(f"\nDone — {total_windows} windows cached")
    print(f"Disk usage: {size_mb:.1f} MB")
    print(f"Failed: {failed}")

print("Caching all participants to local disk ...")
# Change this one line — pass all_participant_info not participant_info
cache_all_participants(all_participant_info, DATA_ROOT, CACHE_DIR)

Caching all participants to local disk ...
  303 — already cached
  304 — already cached
  305 — already cached
  310 — already cached
  312 — already cached
  313 — already cached
  315 — already cached
  316 — already cached
  317 — already cached
  318 — already cached
  319 — already cached
  320 — already cached
  321 — already cached
  322 — already cached
  324 — already cached
  325 — already cached
  326 — already cached
  327 — already cached
  328 — already cached
  330 — already cached
  333 — already cached
  336 — already cached
  338 — already cached
  339 — already cached
  340 — already cached
  341 — already cached
  343 — already cached
  344 — already cached
  345 — already cached
  347 — already cached
  348 — already cached
  350 — already cached
  351 — already cached
  352 — already cached
  353 — already cached
  355 — already cached
  356 — already cached
  357 — already cached
  358 — already cached
  360 — already cached
  362 — already cached
  363 — alread

In [17]:
import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

random.seed(42)

class DiskDAICDataset(Dataset):
    def __init__(self, cache_dir, pids):
        self.cache_dir = cache_dir
        self.samples   = []
        for pid in pids:
            path = os.path.join(cache_dir, f"{pid}.npy")
            if not os.path.exists(path):
                continue
            try:
                data = np.load(path, allow_pickle=True).item()
                n    = len(data['label'])
                for i in range(n):
                    self.samples.append({
                        'path':   path,
                        'idx':    i,
                        'label':  int(data['label'][i]),
                        'gender': int(data['gender'][i]),
                    })
            except Exception as e:
                print(f"  ERROR {pid}: {e}")

        dep  = sum(1 for s in self.samples if s['label'] == 1)
        ctrl = len(self.samples) - dep
        print(f"  {len(pids)} participants → "
              f"{len(self.samples)} windows  "
              f"(dep={dep}  ctrl={ctrl})")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s    = self.samples[idx]
        data = np.load(s['path'], allow_pickle=True).item()
        i    = s['idx']
        return (
            torch.FloatTensor(data['audio'][i]),
            torch.FloatTensor(data['au'][i]),
            torch.FloatTensor(data['landmark'][i]),
            torch.FloatTensor(data['gaze'][i]),
            torch.FloatTensor(data['pose'][i]),
            torch.tensor(s['label'],  dtype=torch.long),
            torch.tensor(s['gender'], dtype=torch.long),
        )


# ── USE OFFICIAL SPLITS — not random split ───────────────────
# This is what every paper does including Gimeno-Gomez 2024
# Train on train split, validate on dev split, test on test split

train_pids = list(train_participant_info.keys())
val_pids   = list(dev_participant_info.keys())
test_pids  = list(test_participant_info.keys())

print("Building train dataset:")
train_dataset = DiskDAICDataset(CACHE_DIR, train_pids)

print("\nBuilding val (dev) dataset:")
val_dataset   = DiskDAICDataset(CACHE_DIR, val_pids)

print("\nBuilding test dataset:")
test_dataset  = DiskDAICDataset(CACHE_DIR, test_pids)

# Val and test loaders — no sampler needed
val_loader = DataLoader(
    val_dataset,  batch_size=32,
    shuffle=False, num_workers=8, pin_memory=True)

test_loader = DataLoader(
    test_dataset, batch_size=16,
    shuffle=False, num_workers=2, pin_memory=True)

# Feature dims from one val batch
_a, _u, _l, _g, _p, _lbl, _gen = next(iter(val_loader))
audio_dim = _a.shape[-1]
au_dim    = _u.shape[-1]
lm_dim    = _l.shape[-1]
gaze_dim  = _g.shape[-1]
pose_dim  = _p.shape[-1]

print(f"\nFeature dims:")
print(f"  audio={audio_dim} au={au_dim} "
      f"lm={lm_dim} gaze={gaze_dim} pose={pose_dim}")
print(f"\nTrain : {len(train_pids)} participants  "
      f"{len(train_dataset)} windows")
print(f"Val   : {len(val_pids)} participants  "
      f"{len(val_dataset)} windows")
print(f"Test  : {len(test_pids)} participants  "
      f"{len(test_dataset)} windows")

Building train dataset:
  107 participants → 16089 windows  (dep=4524  ctrl=11565)

Building val (dev) dataset:
  34 participants → 5688 windows  (dep=2105  ctrl=3583)

Building test dataset:
  44 participants → 7557 windows  (dep=2434  ctrl=5123)

Feature dims:
  audio=79 au=20 lm=204 gaze=12 pose=6

Train : 107 participants  16089 windows
Val   : 34 participants  5688 windows
Test  : 44 participants  7557 windows


In [11]:
import torch
import torch.nn as nn

class ModalityEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
        )
    def forward(self, x):
        B, T, D = x.shape
        return self.net(x.reshape(B*T, D)).reshape(B, T, 256)


class BaselineModel(nn.Module):
    def __init__(self, d_model=256, n_heads=8, n_layers=4,
                 dropout=0.1, num_classes=2):
        super().__init__()
        self.audio_pool = nn.AvgPool1d(kernel_size=4, stride=4)
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=512, dropout=dropout,
            batch_first=True)
        self.transformer = nn.TransformerEncoder(
            enc, num_layers=n_layers)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64, num_classes))

    def build_encoders(self, audio_dim, au_dim,
                       lm_dim, gaze_dim, pose_dim):
        self.enc_audio    = ModalityEncoder(audio_dim)
        self.enc_au       = ModalityEncoder(au_dim)
        self.enc_landmark = ModalityEncoder(lm_dim)
        self.enc_gaze     = ModalityEncoder(gaze_dim)
        self.enc_pose     = ModalityEncoder(pose_dim)

    def forward(self, audio, au, landmark, gaze, pose):
        a = self.enc_audio(audio)
        u = self.enc_au(au)
        l = self.enc_landmark(landmark)
        g = self.enc_gaze(gaze)
        p = self.enc_pose(pose)
        a = self.audio_pool(
            a.permute(0,2,1)).permute(0,2,1)
        fused = a + u + l + g + p
        out   = self.transformer(fused).mean(dim=1)
        return self.classifier(out)

print("ModalityEncoder + BaselineModel defined ✓")

ModalityEncoder + BaselineModel defined ✓


In [13]:
# Cell 9 — rebuild model correctly
# build_encoders THEN move everything to device in one shot

gata_model = GATADepModel(K=4)
gata_model.build_encoders(audio_dim, au_dim, lm_dim, gaze_dim, pose_dim)
gata_model = gata_model.to(device)   # ← move AFTER build_encoders

n_params = sum(p.numel() for p in gata_model.parameters()
               if p.requires_grad)
print(f"GATA-Dep parameters: {n_params:,}")

# Sanity check
with torch.no_grad():
    _out = gata_model(
        _a.to(device), _u.to(device), _l.to(device),
        _g.to(device), _p.to(device), _gen.to(device))
    print(f"Forward pass: {_out.shape} ← OK")

GATA-Dep parameters: 3,526,284
Forward pass: torch.Size([32, 2]) ← OK


In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GATAFusion(nn.Module):
    def __init__(self, d_model=256, n_heads=4, K=4, dropout=0.1):
        super().__init__()
        self.K         = K
        self.n_offsets = 2 * K + 1

        self.attn_au   = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True)
        self.attn_lm   = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True)
        self.attn_gaze = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True)
        self.attn_pose = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True)

        self.offset_weights = nn.Parameter(
            torch.zeros(2, 4, self.n_offsets))

        self.norm_au   = nn.LayerNorm(d_model)
        self.norm_lm   = nn.LayerNorm(d_model)
        self.norm_gaze = nn.LayerNorm(d_model)
        self.norm_pose = nn.LayerNorm(d_model)

        self.output_proj = nn.Sequential(
            nn.Linear(d_model * 4, d_model),
            nn.ReLU(), nn.Dropout(dropout))

    def _shift(self, x, k):
        if k == 0: return x
        B, T, D = x.shape
        out = torch.zeros_like(x)
        if k > 0:
            out[:, k:, :]  = x[:, :T-k, :]
        else:
            ak = abs(k)
            out[:, :T-ak, :] = x[:, ak:, :]
        return out

    def _async_attn(self, query, key_val,
                    attn_mod, gender, mod_idx):
        B, T, D = query.shape
        n = self.n_offsets

        shifted = torch.stack(
            [self._shift(key_val, k)
             for k in range(-self.K, self.K+1)],
            dim=1).reshape(B * n, T, D)

        q_rep = (query.unsqueeze(1)
                      .expand(-1, n, -1, -1)
                      .reshape(B * n, T, D))

        attn_out, _ = attn_mod(
            query=q_rep, key=shifted, value=shifted)
        attn_out = attn_out.view(B, n, T, D)

        w_all    = F.softmax(
            self.offset_weights[:, mod_idx, :], dim=-1)
        w_sample = (w_all[gender]
                        .unsqueeze(-1)
                        .unsqueeze(-1))     # [B, n, 1, 1]

        return (attn_out * w_sample).sum(dim=1)

    def forward(self, audio, au, landmark,
                gaze, pose, gender):
        out_au   = self._async_attn(
            audio, au,       self.attn_au,   gender, 0)
        out_lm   = self._async_attn(
            audio, landmark, self.attn_lm,   gender, 1)
        out_gaze = self._async_attn(
            audio, gaze,     self.attn_gaze, gender, 2)
        out_pose = self._async_attn(
            audio, pose,     self.attn_pose, gender, 3)

        out_au   = self.norm_au  (audio + out_au)
        out_lm   = self.norm_lm  (audio + out_lm)
        out_gaze = self.norm_gaze(audio + out_gaze)
        out_pose = self.norm_pose(audio + out_pose)

        return self.output_proj(
            torch.cat([out_au, out_lm,
                       out_gaze, out_pose], dim=-1))


class GATADepModel(nn.Module):
    def __init__(self, d_model=256, n_heads=4, n_layers=4,
                 K=4, dropout=0.1, num_classes=2):
        super().__init__()
        self.audio_pool = nn.AvgPool1d(kernel_size=4, stride=4)
        self.gata       = GATAFusion(d_model, n_heads, K, dropout)
        self.mc_dropout = nn.Dropout(p=0.3)
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=8,
            dim_feedforward=512, dropout=dropout,
            batch_first=True)
        self.transformer = nn.TransformerEncoder(
            enc, num_layers=n_layers)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64, num_classes))

    def build_encoders(self, audio_dim, au_dim,
                       lm_dim, gaze_dim, pose_dim):
        self.enc_audio    = ModalityEncoder(audio_dim)
        self.enc_au       = ModalityEncoder(au_dim)
        self.enc_landmark = ModalityEncoder(lm_dim)
        self.enc_gaze     = ModalityEncoder(gaze_dim)
        self.enc_pose     = ModalityEncoder(pose_dim)

    def _one_pass(self, a, u, l, g, p, gender):
        a = self.mc_dropout(a); u = self.mc_dropout(u)
        l = self.mc_dropout(l); g = self.mc_dropout(g)
        p = self.mc_dropout(p)
        fused = self.gata(a, u, l, g, p, gender)
        out   = self.transformer(fused).mean(dim=1)
        return self.classifier(out)

    def forward(self, audio, au, landmark,
                gaze, pose, gender, mc_samples=1):
        a = self.enc_audio(audio)
        u = self.enc_au(au)
        l = self.enc_landmark(landmark)
        g = self.enc_gaze(gaze)
        p = self.enc_pose(pose)
        a = self.audio_pool(
            a.permute(0,2,1)).permute(0,2,1)

        if mc_samples == 1:
            return self._one_pass(a, u, l, g, p, gender)

        prev = self.mc_dropout.training
        self.mc_dropout.train()
        logits_list = [
            self._one_pass(a, u, l, g, p, gender)
            for _ in range(mc_samples)
        ]
        self.mc_dropout.training = prev
        stacked = torch.stack(logits_list)
        return stacked.mean(0), stacked.var(0).mean(-1)


# Build + sanity check
# Build + sanity check
gata_model = GATADepModel(K=4)                          # ← no .to(device) here
gata_model.build_encoders(
    audio_dim, au_dim, lm_dim, gaze_dim, pose_dim)
gata_model = gata_model.to(device)                      # ← move to device AFTER build_encoders

n_params = sum(p.numel() for p in gata_model.parameters()
               if p.requires_grad)
print(f"GATA-Dep parameters: {n_params:,}")

with torch.no_grad():
    _out = gata_model(
        _a.to(device), _u.to(device), _l.to(device),
        _g.to(device), _p.to(device), _gen.to(device))
    print(f"Forward pass: {_out.shape} ← OK")

GATA-Dep parameters: 3,526,284
Forward pass: torch.Size([32, 2]) ← OK


In [14]:
# Run this ONCE to delete the old incompatible checkpoint
import os

ckpt_path = os.path.join(OUTPUT_DIR, 'gata_model_ckpt.pt')
if os.path.exists(ckpt_path):
    os.remove(ckpt_path)
    print("Old checkpoint deleted ✓")
else:
    print("No checkpoint found — nothing to delete")

Old checkpoint deleted ✓


In [15]:
# Run this first — find and delete any cached files 
# where landmark has 0 columns
import numpy as np
import os

bad_pids = []
for fname in os.listdir(CACHE_DIR):
    if not fname.endswith('.npy'):
        continue
    path = os.path.join(CACHE_DIR, fname)
    try:
        data = np.load(path, allow_pickle=True).item()
        if data['landmark'].shape[2] == 0:
            print(f"BAD: {fname}  landmark shape: {data['landmark'].shape}")
            bad_pids.append(fname)
            os.remove(path)
    except Exception as e:
        print(f"ERROR {fname}: {e}")
        bad_pids.append(fname)
        os.remove(path)

print(f"\nDeleted {len(bad_pids)} bad cache files: {bad_pids}")
print("Now rerun the cache cell — it will reprocess only deleted files")

BAD: 432.npy  landmark shape: (155, 150, 0)
BAD: 367.npy  landmark shape: (272, 150, 0)
BAD: 396.npy  landmark shape: (130, 150, 0)

Deleted 3 bad cache files: ['432.npy', '367.npy', '396.npy']
Now rerun the cache cell — it will reprocess only deleted files


In [16]:
# Delete the 3 bad participants permanently
import os

bad_files = ['367.npy', '396.npy', '432.npy']

for fname in bad_files:
    path = os.path.join(CACHE_DIR, fname)
    if os.path.exists(path):
        os.remove(path)
        print(f"Deleted: {fname}")

# Remove them from participant info dicts too
for pid in ['367', '396', '432']:
    dev_participant_info.pop(pid,  None)
    test_participant_info.pop(pid, None)
    all_participant_info.pop(pid,  None)

# Rebuild pids lists
val_pids  = list(dev_participant_info.keys())
test_pids = list(test_participant_info.keys())

print(f"\nVal  participants now: {len(val_pids)}")
print(f"Test participants now: {len(test_pids)}")
print("\nSafe to run training now ✓")


Val  participants now: 34
Test participants now: 44

Safe to run training now ✓


In [20]:
import os

OUTPUT_DIR = "/home/user/GATA-Dep/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Exists:", os.path.exists(OUTPUT_DIR))

Exists: True


In [21]:
gata_save = os.path.join(OUTPUT_DIR, 'gata_model.pt')
print(gata_save)

/home/user/GATA-Dep/results/gata_model.pt


In [22]:
print("OUTPUT_DIR =", OUTPUT_DIR)

OUTPUT_DIR = /home/user/GATA-Dep/results


In [69]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import f1_score, precision_score, recall_score


# ── RAM Dataset — loads everything once, then instant access ──
class RAMDAICDataset(Dataset):
    def __init__(self, cache_dir, pids):
        audio_list = []; au_list = []; lm_list = []
        gaze_list  = []; pose_list = []
        labels = []; genders = []

        for pid in pids:
            path = os.path.join(cache_dir, f"{pid}.npy")
            if not os.path.exists(path):
                continue
            try:
                data = np.load(path, allow_pickle=True).item()
                n = len(data['label'])
                audio_list.append(data['audio'])
                au_list.append(data['au'])
                lm_list.append(data['landmark'])
                gaze_list.append(data['gaze'])
                pose_list.append(data['pose'])
                labels.extend(data['label'].tolist())
                genders.extend(data['gender'].tolist())
            except Exception as e:
                print(f"  SKIP {pid}: {e}")

        self.audio    = torch.FloatTensor(np.concatenate(audio_list))
        self.au       = torch.FloatTensor(np.concatenate(au_list))
        self.landmark = torch.FloatTensor(np.concatenate(lm_list))
        self.gaze     = torch.FloatTensor(np.concatenate(gaze_list))
        self.pose     = torch.FloatTensor(np.concatenate(pose_list))
        self.labels   = torch.tensor(labels,  dtype=torch.long)
        self.genders  = torch.tensor(genders, dtype=torch.long)

        dep  = self.labels.sum().item()
        ctrl = len(self.labels) - dep
        ram  = sum(x.nbytes for x in [
            self.audio, self.au, self.landmark,
            self.gaze,  self.pose]) / 1e9
        print(f"  {len(pids)} participants → "
              f"{len(self.labels)} windows  "
              f"dep={dep} ctrl={ctrl}  "
              f"RAM={ram:.2f} GB")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (self.audio[idx], self.au[idx],
                self.landmark[idx], self.gaze[idx],
                self.pose[idx], self.labels[idx],
                self.genders[idx])


# ── Training function using RAM dataset ───────────────────────
def train_gata_full(model, train_pids, val_loader,
                    n_epochs, device, save_path):

    print("Loading train data into RAM...")
    _ds    = RAMDAICDataset(CACHE_DIR, train_pids)
    n_dep  = (_ds.labels == 1).sum().item()
    n_ctrl = (_ds.labels == 0).sum().item()
    total  = len(_ds)
    ratio  = n_ctrl / n_dep

    print(f"Train → dep:{n_dep}  ctrl:{n_ctrl}  ratio:{ratio:.2f}")

    # Balanced sampler
    w_dep  = total / (2 * n_dep)
    w_ctrl = total / (2 * n_ctrl)
    sw = [w_dep if _ds.labels[i].item() == 1 else w_ctrl
          for i in range(total)]
    sampler = WeightedRandomSampler(
        weights=sw, num_samples=total, replacement=True)

    train_loader = DataLoader(
        _ds, batch_size=32,        # 16 is safe with RAM dataset
        sampler=sampler,
        num_workers=8,             # safe now — reading from RAM
        pin_memory=True)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=5e-5, weight_decay=1e-4)

    def lr_lambda(epoch):
        warmup = 5
        if epoch < warmup:
            return (epoch + 1) / warmup
        prog = (epoch - warmup) / max(n_epochs - warmup, 1)
        return 0.5 * (1 + np.cos(np.pi * prog))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler    = GradScaler()

    best_f1 = 0.0; no_improve = 0
    patience = 20; start_epoch = 1
    history  = {'train_loss': [], 'val_f1': [],
                'val_prec': [],   'val_rec': []}

    # Resume from checkpoint
    ckpt_path = save_path.replace('.pt', '_ckpt.pt')
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        scaler.load_state_dict(ckpt['scaler'])
        history     = ckpt['history']
        best_f1     = ckpt['best_f1']
        no_improve  = ckpt['no_improve']
        start_epoch = ckpt['epoch'] + 1
        print(f"Resumed from epoch {ckpt['epoch']} "
              f"(best F1: {best_f1:.4f})")

    print(f"\n{'='*65}")
    print(f"GATA-Dep | K=12 | batch=32| RAM | AMP | "
          f"Warmup=5 | Patience={patience}")
    print(f"{'='*65}")

    for epoch in range(start_epoch, n_epochs + 1):

        # Train
        model.train()
        total_loss, n_batches = 0.0, 0

        for batch in train_loader:
            audio, au, lm, gaze, pose, label, gender = batch
            audio  = audio.to(device,  non_blocking=True)
            au     = au.to(device,     non_blocking=True)
            lm     = lm.to(device,     non_blocking=True)
            gaze   = gaze.to(device,   non_blocking=True)
            pose   = pose.to(device,   non_blocking=True)
            label  = label.to(device,  non_blocking=True)
            gender = gender.to(device, non_blocking=True)

            optimizer.zero_grad()
            with autocast():
                logits = model(audio, au, lm, gaze, pose,
                               gender, mc_samples=1)
                loss   = criterion(logits, label)
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            n_batches  += 1

        scheduler.step()
        avg_loss   = total_loss / max(n_batches, 1)
        current_lr = optimizer.param_groups[0]['lr']

        # Validate
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                audio, au, lm, gaze, pose, label, gender = batch
                with autocast():
                    logits = model(
                        audio.to(device, non_blocking=True),
                        au.to(device,    non_blocking=True),
                        lm.to(device,    non_blocking=True),
                        gaze.to(device,  non_blocking=True),
                        pose.to(device,  non_blocking=True),
                        gender.to(device,non_blocking=True),
                        mc_samples=1)
                all_preds.extend(logits.argmax(1).cpu().tolist())
                all_labels.extend(label.cpu().tolist())

        if not all_labels:
            continue

        f1   = f1_score(all_labels, all_preds,
                        average='binary', zero_division=0)
        prec = precision_score(all_labels, all_preds,
                               average='binary', zero_division=0)
        rec  = recall_score(all_labels, all_preds,
                            average='binary', zero_division=0)

        pred_dep  = sum(all_preds)
        pred_ctrl = len(all_preds) - pred_dep
        true_dep  = sum(all_labels)

        history['train_loss'].append(avg_loss)
        history['val_f1'].append(f1)
        history['val_prec'].append(prec)
        history['val_rec'].append(rec)

        flag = ""
        if f1 > best_f1:
            best_f1    = f1
            no_improve = 0
            torch.save(model.state_dict(), save_path)
            flag = " ← BEST"
        else:
            no_improve += 1

        torch.save({
            'epoch': epoch, 'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'scaler':    scaler.state_dict(),
            'history':   history,
            'best_f1':   best_f1,
            'no_improve':no_improve,
        }, ckpt_path)

        print(f"Ep {epoch:3d}/{n_epochs} | "
              f"Loss {avg_loss:.4f} | "
              f"F1 {f1:.4f} | P {prec:.4f} R {rec:.4f} | "
              f"Pred[dep={pred_dep} ctrl={pred_ctrl} "
              f"true={true_dep}] | "
              f"LR {current_lr:.1e}{flag}")

        if no_improve >= patience:
            print(f"\nEarly stop at epoch {epoch}")
            break

    print(f"\nBest GATA-Dep F1: {best_f1:.4f}")
    return history, best_f1


# ── Also rebuild val loader with RAM dataset ──────────────────
print("Loading val data into RAM...")
val_dataset = RAMDAICDataset(CACHE_DIR, val_pids)
val_loader  = DataLoader(
    val_dataset, batch_size=32,
    shuffle=False, num_workers=8, pin_memory=True)

# ── Run ───────────────────────────────────────────────────────
gata_save = os.path.join(OUTPUT_DIR, 'gata_model.pt')

history_gata, best_f1_gata = train_gata_full(
    model      = gata_model,
    train_pids = train_pids,
    val_loader = val_loader,
    n_epochs   = 60,
    device     = device,
    save_path  = gata_save,
)

Loading val data into RAM...
  34 participants → 5688 windows  dep=2105 ctrl=3583  RAM=1.90 GB
Loading train data into RAM...
  107 participants → 16089 windows  dep=4524 ctrl=11565  RAM=5.39 GB
Train → dep:4524  ctrl:11565  ratio:2.56

GATA-Dep | K=12 | batch=32| RAM | AMP | Warmup=5 | Patience=20
Ep   1/60 | Loss 0.6801 | F1 0.5247 | P 0.4248 R 0.6860 | Pred[dep=3399 ctrl=2289 true=2105] | LR 2.0e-05 ← BEST
Ep   2/60 | Loss 0.6285 | F1 0.5666 | P 0.4225 R 0.8599 | Pred[dep=4284 ctrl=1404 true=2105] | LR 3.0e-05 ← BEST
Ep   3/60 | Loss 0.5382 | F1 0.4985 | P 0.4133 R 0.6280 | Pred[dep=3199 ctrl=2489 true=2105] | LR 4.0e-05
Ep   4/60 | Loss 0.4682 | F1 0.4359 | P 0.4027 R 0.4751 | Pred[dep=2483 ctrl=3205 true=2105] | LR 5.0e-05


KeyboardInterrupt: 

In [82]:
gata_model.load_state_dict(
    torch.load(gata_save, map_location=device)
)
gata_model.eval()

GATADepModel(
  (audio_pool): AvgPool1d(kernel_size=(4,), stride=(4,), padding=(0,))
  (gata): GATAFusion(
    (attn_au): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
    )
    (attn_lm): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
    )
    (attn_gaze): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
    )
    (attn_pose): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
    )
    (norm_au): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (norm_lm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (norm_gaze): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (norm_pose): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (output_proj): Sequential(
      (0): Linear(in_features=

CLAUDE CODE NEW CHECKING


In [19]:
import os

# Delete ALL checkpoints and saved models
for f in os.listdir(OUTPUT_DIR):
    if f.endswith('.pt'):
        os.remove(os.path.join(OUTPUT_DIR, f))
        print(f"Deleted: {f}")

print("All checkpoints deleted")

Deleted: gata_model.pt
All checkpoints deleted


In [23]:
# Rebuild GATA model from scratch — fresh weights
gata_model = GATADepModel(K=12)
gata_model.build_encoders(audio_dim, au_dim,
                           lm_dim, gaze_dim, pose_dim)
gata_model = gata_model.to(device)

total = sum(p.numel() for p in gata_model.parameters())
print(f"Fresh model: {total:,} parameters")

# Confirm weights are random
sample_w = gata_model.gata.offset_weights.data
print(f"Offset weights sample: {sample_w[0,0,:3]}")
# Should show zeros (freshly initialized)

Fresh model: 3,526,412 parameters
Offset weights sample: tensor([0., 0., 0.], device='cuda:0')


In [65]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import f1_score, precision_score, recall_score


class RAMDAICDataset(Dataset):
    def __init__(self, cache_dir, pids):
        audio_list = []; au_list = []; lm_list = []
        gaze_list  = []; pose_list = []
        labels = []; genders = []

        for pid in pids:
            path = os.path.join(cache_dir, f"{pid}.npy")
            if not os.path.exists(path):
                continue
            try:
                data = np.load(path, allow_pickle=True).item()
                audio_list.append(data['audio'])
                au_list.append(data['au'])
                lm_list.append(data['landmark'])
                gaze_list.append(data['gaze'])
                pose_list.append(data['pose'])
                labels.extend(data['label'].tolist())
                genders.extend(data['gender'].tolist())
            except Exception as e:
                print(f"  SKIP {pid}: {e}")

        self.audio    = torch.FloatTensor(np.concatenate(audio_list))
        self.au       = torch.FloatTensor(np.concatenate(au_list))
        self.landmark = torch.FloatTensor(np.concatenate(lm_list))
        self.gaze     = torch.FloatTensor(np.concatenate(gaze_list))
        self.pose     = torch.FloatTensor(np.concatenate(pose_list))
        self.labels   = torch.tensor(labels,  dtype=torch.long)
        self.genders  = torch.tensor(genders, dtype=torch.long)

        dep  = self.labels.sum().item()
        ctrl = len(self.labels) - dep
        ram  = sum(x.element_size() * x.nelement()
                   for x in [self.audio, self.au,
                              self.landmark,
                              self.gaze, self.pose]) / 1e9
        print(f"  {len(pids)} participants → "
              f"{len(self.labels)} windows  "
              f"dep={dep} ctrl={ctrl}  "
              f"RAM={ram:.2f} GB")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (self.audio[idx], self.au[idx],
                self.landmark[idx], self.gaze[idx],
                self.pose[idx], self.labels[idx],
                self.genders[idx])


def train_gata_fixed(model, train_pids, val_pids,
                     cache_dir, n_epochs, device, save_path):

    # ── Load data into RAM ────────────────────────────────────
    print("Loading train data into RAM...")
    train_ds = RAMDAICDataset(cache_dir, train_pids)
    print("Loading val data into RAM...")
    val_ds   = RAMDAICDataset(cache_dir, val_pids)

    n_dep  = (train_ds.labels == 1).sum().item()
    n_ctrl = (train_ds.labels == 0).sum().item()
    total  = len(train_ds)
    ratio  = n_ctrl / n_dep
    print(f"\nTrain → dep:{n_dep} ctrl:{n_ctrl} ratio:{ratio:.2f}")

    # ── Balanced sampler ──────────────────────────────────────
    w_dep  = total / (2 * n_dep)
    w_ctrl = total / (2 * n_ctrl)
    sw = [w_dep if train_ds.labels[i].item() == 1
          else w_ctrl for i in range(total)]
    sampler = WeightedRandomSampler(
        weights=sw, num_samples=total, replacement=True)

    train_loader = DataLoader(
        train_ds, batch_size=32,
        sampler=sampler,
        num_workers=4,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=32,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    # ── Loss — class weight ON TOP of sampler ─────────────────
    # Both together give strongest signal to learn depressed class
    class_weight = torch.tensor([1.0, ratio]).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weight)

    # ── TWO separate optimizers ───────────────────────────────
    # Encoders need higher LR to learn features
    # GATA offset weights need LOWER LR to stabilize
    encoder_params = []
    gata_params    = []
    other_params   = []

    for name, param in model.named_parameters():
        if 'offset_weights' in name:
            gata_params.append(param)
        elif 'enc_' in name:
            encoder_params.append(param)
        else:
            other_params.append(param)

    optimizer = torch.optim.AdamW([
        {'params': encoder_params,
         'lr': 3e-5,  'weight_decay': 1e-4},
        {'params': gata_params,
         'lr': 1e-5,  'weight_decay': 0.0},  # very low LR for offsets
        {'params': other_params,
         'lr': 3e-5,  'weight_decay': 1e-4},
    ])

    # ── Warmup then cosine decay ──────────────────────────────
    def lr_lambda(epoch):
        warmup = 15  # longer warmup for stability
        if epoch < warmup:
            return (epoch + 1) / warmup
        prog = (epoch - warmup) / max(n_epochs - warmup, 1)
        return max(0.1, 0.5 * (1 + np.cos(np.pi * prog)))
        # min LR = 10% of base — never goes to zero

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda)
    scaler = GradScaler()

    best_f1    = 0.0
    no_improve = 0
    patience   = 20
    start_epoch = 1
    history = {
        'train_loss': [], 'val_f1': [],
        'val_prec': [], 'val_rec': []
    }

    # ── Resume from checkpoint ────────────────────────────────
    ckpt_path = save_path.replace('.pt', '_ckpt.pt')
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        try:
            model.load_state_dict(ckpt['model'])
            optimizer.load_state_dict(ckpt['optimizer'])
            scheduler.load_state_dict(ckpt['scheduler'])
            scaler.load_state_dict(ckpt['scaler'])
            history     = ckpt['history']
            best_f1     = ckpt['best_f1']
            no_improve  = ckpt['no_improve']
            start_epoch = ckpt['epoch'] + 1
            print(f"Resumed from epoch {ckpt['epoch']} "
                  f"(best F1: {best_f1:.4f})")
        except Exception as e:
            print(f"Checkpoint load failed: {e} — starting fresh")

    print(f"\n{'='*65}")
    print(f"GATA-Dep FIXED | K=12 | batch=32 | AMP | "
          f"Warmup=8 | Patience={patience}")
    print(f"LR: encoders=1e-4 | offset_weights=1e-5 | "
          f"transformer=1e-4")
    print(f"{'='*65}")

    for epoch in range(start_epoch, n_epochs + 1):

        # ── Train ─────────────────────────────────────────────
        model.train()
        total_loss = 0.0
        n_batches  = 0

        for batch in train_loader:
            (audio, au, lm, gaze,
             pose, label, gender) = batch

            audio  = audio.to(device,  non_blocking=True)
            au     = au.to(device,     non_blocking=True)
            lm     = lm.to(device,     non_blocking=True)
            gaze   = gaze.to(device,   non_blocking=True)
            pose   = pose.to(device,   non_blocking=True)
            label  = label.to(device,  non_blocking=True)
            gender = gender.to(device, non_blocking=True)

            optimizer.zero_grad()
            with autocast():
                logits = model(
                    audio, au, lm, gaze, pose,
                    gender, mc_samples=1)
                loss = criterion(logits, label)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=0.5)  # tighter clipping
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            n_batches  += 1

        scheduler.step()
        avg_loss   = total_loss / max(n_batches, 1)
        current_lr = optimizer.param_groups[0]['lr']

        # ── Validate ──────────────────────────────────────────
        model.eval()
        all_preds  = []
        all_labels = []

        with torch.no_grad():
            for batch in val_loader:
                (audio, au, lm, gaze,
                 pose, label, gender) = batch
                with autocast():
                    logits = model(
                        audio.to(device,  non_blocking=True),
                        au.to(device,     non_blocking=True),
                        lm.to(device,     non_blocking=True),
                        gaze.to(device,   non_blocking=True),
                        pose.to(device,   non_blocking=True),
                        gender.to(device, non_blocking=True),
                        mc_samples=1)
                all_preds.extend(
                    logits.argmax(1).cpu().tolist())
                all_labels.extend(label.cpu().tolist())

        if not all_labels:
            continue

        f1   = f1_score(all_labels, all_preds,
                        average='binary', zero_division=0)
        prec = precision_score(all_labels, all_preds,
                               average='binary', zero_division=0)
        rec  = recall_score(all_labels, all_preds,
                            average='binary', zero_division=0)

        pred_dep  = sum(all_preds)
        pred_ctrl = len(all_preds) - pred_dep
        true_dep  = sum(all_labels)

        history['train_loss'].append(avg_loss)
        history['val_f1'].append(f1)
        history['val_prec'].append(prec)
        history['val_rec'].append(rec)

        flag = ""
        if f1 > best_f1:
            best_f1    = f1
            no_improve = 0
            torch.save(model.state_dict(), save_path)
            flag = " ← BEST"
        else:
            no_improve += 1

        # Save checkpoint every epoch
        torch.save({
            'epoch':      epoch,
            'model':      model.state_dict(),
            'optimizer':  optimizer.state_dict(),
            'scheduler':  scheduler.state_dict(),
            'scaler':     scaler.state_dict(),
            'history':    history,
            'best_f1':    best_f1,
            'no_improve': no_improve,
        }, ckpt_path)

        # Print offset weights every 10 epochs
        # So you can see what the model is learning
        if epoch % 10 == 0 or epoch <= 3:
            with torch.no_grad():
                w = torch.softmax(
                    model.gata.offset_weights, dim=-1)
                male_au   = w[0, 0].argmax().item()
                female_au = w[1, 0].argmax().item()
                K = model.gata.K
                male_sec   = (male_au   - K) / 25
                female_sec = (female_au - K) / 25
                print(f"  Offset → male:{male_sec:+.2f}s "
                      f"female:{female_sec:+.2f}s")

        print(f"Ep {epoch:3d}/{n_epochs} | "
              f"Loss {avg_loss:.4f} | "
              f"F1 {f1:.4f} | "
              f"P {prec:.4f} R {rec:.4f} | "
              f"Pred[dep={pred_dep} ctrl={pred_ctrl} "
              f"true={true_dep}] | "
              f"LR {current_lr:.1e}"
              f"{flag}")

        if no_improve >= patience:
            print(f"\nEarly stop at epoch {epoch} — "
                  f"no improvement for {patience} epochs")
            break

    print(f"\nBest GATA-Dep F1: {best_f1:.4f}")
    return history, best_f1


# ── Run ───────────────────────────────────────────────────────
# Delete old checkpoint first to start fresh
ckpt_path = os.path.join(OUTPUT_DIR, 'gata_model_ckpt.pt')
if os.path.exists(ckpt_path):
    os.remove(ckpt_path)
    print("Old checkpoint deleted")

gata_save = os.path.join(OUTPUT_DIR, 'gata_model.pt')

history_gata, best_f1_gata = train_gata_fixed(
    model       = gata_model,
    train_pids  = train_pids,
    val_pids    = val_pids,
    cache_dir   = CACHE_DIR,
    n_epochs    = 80,
    device      = device,
    save_path   = gata_save,
)

print(f"\n{'='*55}")
print(f"FINAL RESULT")
print(f"{'='*55}")
print(f"Gimeno-Gomez 2024 (SOTA): F1 = 0.67")
print(f"GATA-Dep (Ours):          F1 = {best_f1_gata:.4f}")
imp = (best_f1_gata - 0.67) * 100
print(f"Improvement over SOTA:    {imp:+.2f} points")

Loading train data into RAM...
  107 participants → 16089 windows  dep=4524 ctrl=11565  RAM=5.39 GB
Loading val data into RAM...
  34 participants → 5688 windows  dep=2105 ctrl=3583  RAM=1.90 GB

Train → dep:4524 ctrl:11565 ratio:2.56

GATA-Dep FIXED | K=12 | batch=32 | AMP | Warmup=8 | Patience=20
LR: encoders=1e-4 | offset_weights=1e-5 | transformer=1e-4
  Offset → male:+0.04s female:+0.48s
Ep   1/80 | Loss 0.6023 | F1 0.5402 | P 0.3701 R 1.0000 | Pred[dep=5688 ctrl=0 true=2105] | LR 4.0e-06 ← BEST
  Offset → male:+0.00s female:-0.48s
Ep   2/80 | Loss 0.5888 | F1 0.5403 | P 0.3701 R 1.0000 | Pred[dep=5687 ctrl=1 true=2105] | LR 6.0e-06 ← BEST
  Offset → male:+0.00s female:-0.48s
Ep   3/80 | Loss 0.5578 | F1 0.5616 | P 0.3923 R 0.9881 | Pred[dep=5302 ctrl=386 true=2105] | LR 8.0e-06 ← BEST
Ep   4/80 | Loss 0.5069 | F1 0.5756 | P 0.4284 R 0.8770 | Pred[dep=4309 ctrl=1379 true=2105] | LR 1.0e-05 ← BEST
Ep   5/80 | Loss 0.4510 | F1 0.5538 | P 0.4190 R 0.8166 | Pred[dep=4103 ctrl=1585 tru

KeyboardInterrupt: 

it is running at participant level for validation same as the paper did


In [87]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import f1_score, precision_score, recall_score
from collections import defaultdict

# ── RAM Dataset — loads everything once, then instant access ──
class RAMDAICDataset(Dataset):
    def __init__(self, cache_dir, pids):
        audio_list = []; au_list = []; lm_list = []
        gaze_list  = []; pose_list = []
        labels = []; genders = []
        pids_all = []

        for pid in pids:
            path = os.path.join(cache_dir, f"{pid}.npy")
            if not os.path.exists(path):
                continue
            try:
                data = np.load(path, allow_pickle=True).item()
                n = len(data['label'])
                pids_all.extend([pid] * n)
                audio_list.append(data['audio'])
                au_list.append(data['au'])
                lm_list.append(data['landmark'])
                gaze_list.append(data['gaze'])
                pose_list.append(data['pose'])
                labels.extend(data['label'].tolist())
                genders.extend(data['gender'].tolist())
            except Exception as e:
                print(f"  SKIP {pid}: {e}")

        self.audio    = torch.FloatTensor(np.concatenate(audio_list))
        self.au       = torch.FloatTensor(np.concatenate(au_list))
        self.landmark = torch.FloatTensor(np.concatenate(lm_list))
        self.gaze     = torch.FloatTensor(np.concatenate(gaze_list))
        self.pose     = torch.FloatTensor(np.concatenate(pose_list))
        self.labels   = torch.tensor(labels,  dtype=torch.long)
        self.genders  = torch.tensor(genders, dtype=torch.long)
        self.pids = np.array(pids_all, dtype=np.int64)

        dep  = self.labels.sum().item()
        ctrl = len(self.labels) - dep
        ram  = sum(x.nbytes for x in [
            self.audio, self.au, self.landmark,
            self.gaze,  self.pose]) / 1e9
        print(f"  {len(pids)} participants → "
              f"{len(self.labels)} windows  "
              f"dep={dep} ctrl={ctrl}  "
              f"RAM={ram:.2f} GB")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (self.audio[idx], self.au[idx],
                self.landmark[idx], self.gaze[idx],
                self.pose[idx], self.labels[idx],
                self.genders[idx],self.pids[idx])


# ── Training function using RAM dataset ───────────────────────
def train_gata_full(model, train_pids, val_loader,
                    n_epochs, device, save_path):

    print("Loading train data into RAM...")
    _ds    = RAMDAICDataset(CACHE_DIR, train_pids)
    n_dep  = (_ds.labels == 1).sum().item()
    n_ctrl = (_ds.labels == 0).sum().item()
    total  = len(_ds)
    ratio  = n_ctrl / n_dep

    print(f"Train → dep:{n_dep}  ctrl:{n_ctrl}  ratio:{ratio:.2f}")

    # Balanced sampler
    w_dep  = total / (2 * n_dep)
    w_ctrl = total / (2 * n_ctrl)
    sw = [w_dep if _ds.labels[i].item() == 1 else w_ctrl
          for i in range(total)]
    sampler = WeightedRandomSampler(
        weights=sw, num_samples=total, replacement=True)

    train_loader = DataLoader(
        _ds, batch_size=32,        # 16 is safe with RAM dataset
        sampler=sampler,
        num_workers=8,             # safe now — reading from RAM
        pin_memory=True)

    weight = torch.tensor([1.0, 2.5], device=device)
    criterion = nn.CrossEntropyLoss(weight=weight)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=1e-4, weight_decay=1e-5)

    def lr_lambda(epoch):
        warmup = 5
        if epoch < warmup:
            return (epoch + 1) / warmup
        prog = (epoch - warmup) / max(n_epochs - warmup, 1)
        return 0.5 * (1 + np.cos(np.pi * prog))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler    = GradScaler()

    best_f1 = 0.0; no_improve = 0
    patience = 20; start_epoch = 1
    history  = {'train_loss': [], 'val_f1': [],
                'val_prec': [],   'val_rec': []}

    # Resume from checkpoint
    ckpt_path = save_path.replace('.pt', '_ckpt.pt')
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        scaler.load_state_dict(ckpt['scaler'])
        history     = ckpt['history']
        best_f1     = ckpt['best_f1']
        no_improve  = ckpt['no_improve']
        start_epoch = ckpt['epoch'] + 1
        print(f"Resumed from epoch {ckpt['epoch']} "
              f"(best F1: {best_f1:.4f})")

    print(f"\n{'='*65}")
    print(f"GATA-Dep | K=12 | batch=32| RAM | AMP | "
          f"Warmup=5 | Patience={patience}")
    print(f"{'='*65}")

    for epoch in range(start_epoch, n_epochs + 1):

        # Train
        model.train()
        total_loss, n_batches = 0.0, 0

        for batch in train_loader:
            audio, au, lm, gaze, pose, label, gender,pid = batch
            audio  = audio.to(device,  non_blocking=True)
            au     = au.to(device,     non_blocking=True)
            lm     = lm.to(device,     non_blocking=True)
            gaze   = gaze.to(device,   non_blocking=True)
            pose   = pose.to(device,   non_blocking=True)
            label  = label.to(device,  non_blocking=True)
            gender = gender.to(device, non_blocking=True)

            optimizer.zero_grad()
            with autocast():
                logits = model(audio, au, lm, gaze, pose,
                               gender, mc_samples=1)
                loss   = criterion(logits, label)
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            n_batches  += 1

        scheduler.step()
        avg_loss   = total_loss / max(n_batches, 1)
        current_lr = optimizer.param_groups[0]['lr']

        # Validate
        model.eval()
        pid_probs  = defaultdict(list)
        pid_labels = {}

        with torch.no_grad():
            for batch in val_loader:
                audio, au, lm, gaze, pose, label, gender, pid = batch
                with autocast():
                    logits = model(
                        audio.to(device, non_blocking=True),
                        au.to(device, non_blocking=True),
                        lm.to(device, non_blocking=True),
                        gaze.to(device, non_blocking=True),
                        pose.to(device, non_blocking=True),
                        gender.to(device, non_blocking=True),
                        mc_samples=1
                    )
                    probs = torch.softmax(logits, dim=1)[:, 1]

                for p, pr, y in zip(pid.numpy(), probs.cpu().numpy(), label.numpy()):
                    pid_probs[int(p)].append(float(pr))
                    pid_labels[int(p)] = int(y)

        participant_preds  = []
        participant_labels = []
        for pid in pid_probs:
            mean_prob = np.mean(pid_probs[pid])
            participant_preds.append(1 if mean_prob >= 0.4 else 0)
            participant_labels.append(pid_labels[pid])

        f1   = f1_score(participant_labels, participant_preds, average='binary', zero_division=0)
        prec = precision_score(participant_labels, participant_preds, average='binary', zero_division=0)
        rec  = recall_score(participant_labels, participant_preds, average='binary', zero_division=0)

        pred_dep  = sum(participant_preds)
        pred_ctrl = len(participant_preds) - pred_dep
        true_dep  = sum(participant_labels)
        history['train_loss'].append(avg_loss)
        history['val_f1'].append(f1)
        history['val_prec'].append(prec)
        history['val_rec'].append(rec)

        flag = ""
        if f1 > best_f1:
            best_f1    = f1
            no_improve = 0
            torch.save(model.state_dict(), save_path)
            flag = " ← BEST"
        else:
            no_improve += 1

        torch.save({
            'epoch': epoch, 'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'scaler':    scaler.state_dict(),
            'history':   history,
            'best_f1':   best_f1,
            'no_improve':no_improve,
        }, ckpt_path)

        print(f"Ep {epoch:3d}/{n_epochs} | "
              f"Loss {avg_loss:.4f} | "
              f"F1 {f1:.4f} | P {prec:.4f} R {rec:.4f} | "
              f"Pred[dep={pred_dep} ctrl={pred_ctrl} "
              f"true={true_dep}] | "
              f"LR {current_lr:.1e}{flag}")

        if no_improve >= patience:
            print(f"\nEarly stop at epoch {epoch}")
            break

    print(f"\nBest GATA-Dep F1: {best_f1:.4f}")
    return history, best_f1


# ── Also rebuild val loader with RAM dataset ──────────────────
print("Loading val data into RAM...")
val_dataset = RAMDAICDataset(CACHE_DIR, val_pids)
val_loader  = DataLoader(
    val_dataset, batch_size=32,
    shuffle=False, num_workers=8, pin_memory=True)

# ── Run ───────────────────────────────────────────────────────
gata_save = os.path.join(OUTPUT_DIR, 'gata_model.pt')

history_gata, best_f1_gata = train_gata_full(
    model      = gata_model,
    train_pids = train_pids,
    val_loader = val_loader,
    n_epochs   = 60,
    device     = device,
    save_path  = gata_save,
)

Loading val data into RAM...
  34 participants → 5688 windows  dep=2105 ctrl=3583  RAM=1.90 GB
Loading train data into RAM...
  107 participants → 16089 windows  dep=4524 ctrl=11565  RAM=5.39 GB
Train → dep:4524  ctrl:11565  ratio:2.56
Resumed from epoch 16 (best F1: 0.5666)

GATA-Dep | K=12 | batch=32| RAM | AMP | Warmup=5 | Patience=20
Ep  17/60 | Loss 0.1326 | F1 0.1818 | P 0.1818 R 0.1818 | Pred[dep=11 ctrl=23 true=11] | LR 4.4e-05


KeyboardInterrupt: 

1 june


In [23]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import f1_score, precision_score, recall_score
from collections import defaultdict


class RAMDAICDataset(Dataset):
    def __init__(self, cache_dir, pids):
        audio_list = []; au_list = []; lm_list = []
        gaze_list  = []; pose_list = []
        labels = []; genders = []; pids_all = []

        for pid in pids:
            path = os.path.join(cache_dir, f"{pid}.npy")
            if not os.path.exists(path):
                continue
            try:
                data = np.load(path, allow_pickle=True).item()
                n = len(data['label'])
                pids_all.extend([pid] * n)
                audio_list.append(data['audio'])
                au_list.append(data['au'])
                lm_list.append(data['landmark'])
                gaze_list.append(data['gaze'])
                pose_list.append(data['pose'])
                labels.extend(data['label'].tolist())
                genders.extend(data['gender'].tolist())
            except Exception as e:
                print(f"  SKIP {pid}: {e}")

        self.audio    = torch.FloatTensor(np.concatenate(audio_list))
        self.au       = torch.FloatTensor(np.concatenate(au_list))
        self.landmark = torch.FloatTensor(np.concatenate(lm_list))
        self.gaze     = torch.FloatTensor(np.concatenate(gaze_list))
        self.pose     = torch.FloatTensor(np.concatenate(pose_list))
        self.labels   = torch.tensor(labels,  dtype=torch.long)
        self.genders  = torch.tensor(genders, dtype=torch.long)
        self.pids     = np.array(pids_all,    dtype=np.int64)

        dep  = self.labels.sum().item()
        ctrl = len(self.labels) - dep
        print(f"  {len(pids)} participants → "
              f"{len(self.labels)} windows | "
              f"dep={dep} ctrl={ctrl}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (self.audio[idx], self.au[idx],
                self.landmark[idx], self.gaze[idx],
                self.pose[idx], self.labels[idx],
                self.genders[idx], self.pids[idx])


def evaluate_participant_level(model, loader, device,
                                threshold=0.5):
    """
    Participant-level evaluation exactly matching the paper.
    Aggregate window probabilities per participant then vote.
    """
    model.eval()
    pid_probs  = defaultdict(list)
    pid_labels = {}

    with torch.no_grad():
        for batch in loader:
            audio, au, lm, gaze, pose, label, gender, pid = batch
            with autocast():
                logits = model(
                    audio.to(device,  non_blocking=True),
                    au.to(device,     non_blocking=True),
                    lm.to(device,     non_blocking=True),
                    gaze.to(device,   non_blocking=True),
                    pose.to(device,   non_blocking=True),
                    gender.to(device, non_blocking=True),
                    mc_samples=1
                )
            probs = torch.softmax(logits, dim=1)[:, 1]
            for p, pr, y in zip(
                pid.numpy(),
                probs.cpu().numpy(),
                label.numpy()
            ):
                pid_probs[int(p)].append(float(pr))
                pid_labels[int(p)] = int(y)

    preds  = []
    labels = []
    for p in pid_probs:
        mean_prob = np.mean(pid_probs[p])
        preds.append(1 if mean_prob >= threshold else 0)
        labels.append(pid_labels[p])

    f1   = f1_score(labels, preds,
                    average='binary', zero_division=0)
    prec = precision_score(labels, preds,
                           average='binary', zero_division=0)
    rec  = recall_score(labels, preds,
                        average='binary', zero_division=0)

    n_pred_dep = sum(preds)
    n_true_dep = sum(labels)

    return f1, prec, rec, n_pred_dep, n_true_dep


def find_best_threshold(model, loader, device):
    """
    Find optimal threshold on val set.
    Try 0.1 to 0.9 in steps of 0.02.
    """
    model.eval()
    pid_probs  = defaultdict(list)
    pid_labels = {}

    with torch.no_grad():
        for batch in loader:
            audio, au, lm, gaze, pose, label, gender, pid = batch
            with autocast():
                logits = model(
                    audio.to(device,  non_blocking=True),
                    au.to(device,     non_blocking=True),
                    lm.to(device,     non_blocking=True),
                    gaze.to(device,   non_blocking=True),
                    pose.to(device,   non_blocking=True),
                    gender.to(device, non_blocking=True),
                    mc_samples=1
                )
            probs = torch.softmax(logits, dim=1)[:, 1]
            for p, pr, y in zip(
                pid.numpy(),
                probs.cpu().numpy(),
                label.numpy()
            ):
                pid_probs[int(p)].append(float(pr))
                pid_labels[int(p)] = int(y)

    best_f1   = 0.0
    best_thresh = 0.5
    labels_all = [pid_labels[p] for p in pid_probs]

    for thresh in np.arange(0.1, 0.91, 0.02):
        preds = [1 if np.mean(pid_probs[p]) >= thresh else 0
                 for p in pid_probs]
        f1 = f1_score(labels_all, preds,
                      average='binary', zero_division=0)
        if f1 > best_f1:
            best_f1     = f1
            best_thresh = thresh

    return best_thresh, best_f1


def train_gata_final(model, train_pids, val_pids,
                     cache_dir, n_epochs, device, save_path):

    # Load data into RAM
    print("Loading train data into RAM...")
    train_ds = RAMDAICDataset(cache_dir, train_pids)
    print("Loading val data into RAM...")
    val_ds   = RAMDAICDataset(cache_dir, val_pids)

    n_dep  = (train_ds.labels == 1).sum().item()
    n_ctrl = (train_ds.labels == 0).sum().item()
    total  = len(train_ds)
    ratio  = n_ctrl / n_dep
    print(f"Train → dep:{n_dep} ctrl:{n_ctrl} ratio:{ratio:.2f}")

    # Weighted sampler — balanced batches
    w_dep  = total / (2 * n_dep)
    w_ctrl = total / (2 * n_ctrl)
    sw = [w_dep if train_ds.labels[i].item() == 1
          else w_ctrl for i in range(total)]
    sampler = WeightedRandomSampler(
        weights=sw, num_samples=total, replacement=True)

    train_loader = DataLoader(
        train_ds, batch_size=32,
        sampler=sampler,
        num_workers=4,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=32,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    # No class weight — sampler handles balance
    criterion = nn.CrossEntropyLoss()

    # Separate LRs for encoder vs GATA offset weights
    encoder_params = []
    gata_params    = []
    other_params   = []

    for name, param in model.named_parameters():
        if 'offset_weights' in name:
            gata_params.append(param)
        elif 'enc_' in name:
            encoder_params.append(param)
        else:
            other_params.append(param)

    optimizer = torch.optim.AdamW([
        {'params': encoder_params,
         'lr': 1e-4, 'weight_decay': 1e-4},
        {'params': gata_params,
         'lr': 5e-4, 'weight_decay': 0.0},
        {'params': other_params,
         'lr': 1e-4, 'weight_decay': 1e-4},
    ])

    # Warmup 10 epochs then cosine
    def lr_lambda(epoch):
        warmup = 10
        if epoch < warmup:
            return (epoch + 1) / warmup
        prog = (epoch - warmup) / max(n_epochs - warmup, 1)
        return max(0.1, 0.5 * (1 + np.cos(np.pi * prog)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda)
    scaler = GradScaler()

    best_f1     = 0.0
    no_improve  = 0
    patience    = 20
    start_epoch = 1
    best_thresh = 0.5
    history = {
        'train_loss': [], 'val_f1': [],
        'val_prec': [], 'val_rec': []
    }

    # Resume checkpoint
    ckpt_path = save_path.replace('.pt', '_ckpt.pt')
    if os.path.exists(ckpt_path):
        try:
            ckpt = torch.load(
                ckpt_path, map_location=device,
                weights_only=False)
            model.load_state_dict(ckpt['model'])
            optimizer.load_state_dict(ckpt['optimizer'])
            scheduler.load_state_dict(ckpt['scheduler'])
            scaler.load_state_dict(ckpt['scaler'])
            history     = ckpt['history']
            best_f1     = ckpt['best_f1']
            no_improve  = ckpt['no_improve']
            start_epoch = ckpt['epoch'] + 1
            best_thresh = ckpt.get('best_thresh', 0.5)
            print(f"Resumed from epoch {ckpt['epoch']} "
                  f"best F1={best_f1:.4f} "
                  f"thresh={best_thresh:.2f}")
        except Exception as e:
            print(f"Checkpoint failed: {e} — fresh start")

    print(f"\n{'='*65}")
    print(f"GATA-Dep FINAL | K=12 | batch=32 | "
          f"Warmup=10 | Patience={patience}")
    print(f"{'='*65}")

    for epoch in range(start_epoch, n_epochs + 1):

        # Train
        model.train()
        total_loss = 0.0
        n_batches  = 0

        for batch in train_loader:
            (audio, au, lm, gaze,
             pose, label, gender, pid) = batch

            audio  = audio.to(device,  non_blocking=True)
            au     = au.to(device,     non_blocking=True)
            lm     = lm.to(device,     non_blocking=True)
            gaze   = gaze.to(device,   non_blocking=True)
            pose   = pose.to(device,   non_blocking=True)
            label  = label.to(device,  non_blocking=True)
            gender = gender.to(device, non_blocking=True)

            optimizer.zero_grad()
            with autocast():
                logits = model(
                    audio, au, lm, gaze, pose,
                    gender, mc_samples=1)
                loss = criterion(logits, label)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            n_batches  += 1

        scheduler.step()
        avg_loss   = total_loss / max(n_batches, 1)
        current_lr = optimizer.param_groups[0]['lr']

        # Find best threshold every 5 epochs
        if epoch % 5 == 0 or epoch <= 3:
            best_thresh, _ = find_best_threshold(
                model, val_loader, device)

        # Evaluate at participant level
        f1, prec, rec, n_pred_dep, n_true_dep = \
            evaluate_participant_level(
                model, val_loader, device, best_thresh)

        # Print offset weights
        with torch.no_grad():
            w = torch.softmax(
                model.gata.offset_weights, dim=-1)
            K = model.gata.K
            male_sec   = (w[0,0].argmax().item() - K) / 25
            female_sec = (w[1,0].argmax().item() - K) / 25

        history['train_loss'].append(avg_loss)
        history['val_f1'].append(f1)
        history['val_prec'].append(prec)
        history['val_rec'].append(rec)

        flag = ""
        if f1 > best_f1:
            best_f1    = f1
            no_improve = 0
            torch.save(model.state_dict(), save_path)
            flag = " ← BEST"
        else:
            no_improve += 1

        torch.save({
            'epoch':       epoch,
            'model':       model.state_dict(),
            'optimizer':   optimizer.state_dict(),
            'scheduler':   scheduler.state_dict(),
            'scaler':      scaler.state_dict(),
            'history':     history,
            'best_f1':     best_f1,
            'no_improve':  no_improve,
            'best_thresh': best_thresh,
        }, ckpt_path)

        print(f"Ep {epoch:3d}/{n_epochs} | "
              f"Loss {avg_loss:.4f} | "
              f"F1 {f1:.4f} | P {prec:.4f} R {rec:.4f} | "
              f"Pred[dep={n_pred_dep} true={n_true_dep}] | "
              f"Thr={best_thresh:.2f} | "
              f"Off[M={male_sec:+.2f}s F={female_sec:+.2f}s] | "
              f"LR {current_lr:.1e}{flag}")

        if no_improve >= patience:
            print(f"\nEarly stop at epoch {epoch}")
            break

    print(f"\nBest GATA-Dep F1: {best_f1:.4f} "
          f"at threshold {best_thresh:.2f}")
    return history, best_f1


# ── Delete old checkpoint and run fresh ──────────────────────
for f in os.listdir(OUTPUT_DIR):
    if f.endswith('.pt'):
        os.remove(os.path.join(OUTPUT_DIR, f))
        print(f"Deleted: {f}")

# Rebuild fresh model
gata_model = GATADepModel(K=12)
gata_model.build_encoders(
    audio_dim, au_dim, lm_dim, gaze_dim, pose_dim)
gata_model = gata_model.to(device)
print(f"Fresh model: "
      f"{sum(p.numel() for p in gata_model.parameters()):,} params")

gata_save = os.path.join(OUTPUT_DIR, 'gata_model.pt')

history_gata, best_f1_gata = train_gata_final(
    model       = gata_model,
    train_pids  = train_pids,
    val_pids    = val_pids,
    cache_dir   = CACHE_DIR,
    n_epochs    = 80,
    device      = device,
    save_path   = gata_save,
)

print(f"\n{'='*55}")
print(f"FINAL RESULT")
print(f"{'='*55}")
print(f"Gimeno-Gomez 2024 (SOTA): 0.67")
print(f"GATA-Dep (Ours):          {best_f1_gata:.4f}")
print(f"Improvement:              "
      f"{(best_f1_gata-0.67)*100:+.2f} pts")

Deleted: ablation_K4_ckpt.pt
Deleted: ablation_K8_ckpt.pt
Deleted: gata_model_ckpt.pt
Deleted: ablation_K8.pt
Deleted: ablation_K4.pt
Deleted: gata_model.pt
Fresh model: 3,526,412 params
Loading train data into RAM...
  107 participants → 16089 windows | dep=4524 ctrl=11565
Loading val data into RAM...
  34 participants → 5688 windows | dep=2105 ctrl=3583
Train → dep:4524 ctrl:11565 ratio:2.56

GATA-Dep FINAL | K=12 | batch=32 | Warmup=10 | Patience=20
Ep   1/80 | Loss 0.6668 | F1 0.5882 | P 0.4348 R 0.9091 | Pred[dep=23 true=11] | Thr=0.52 | Off[M=+0.00s F=+0.48s] | LR 2.0e-05 ← BEST
Ep   2/80 | Loss 0.5711 | F1 0.5789 | P 0.4074 R 1.0000 | Pred[dep=27 true=11] | Thr=0.24 | Off[M=-0.12s F=-0.20s] | LR 3.0e-05
Ep   3/80 | Loss 0.4781 | F1 0.5946 | P 0.4231 R 1.0000 | Pred[dep=26 true=11] | Thr=0.30 | Off[M=-0.24s F=-0.04s] | LR 4.0e-05 ← BEST
Ep   4/80 | Loss 0.4053 | F1 0.4848 | P 0.3636 R 0.7273 | Pred[dep=22 true=11] | Thr=0.30 | Off[M=-0.20s F=-0.04s] | LR 5.0e-05
Ep   5/80 | Loss 

KeyboardInterrupt: 

In [36]:
import torch

torch.save(
    gata_model.state_dict(),
    "./results/GATA_best_F1_0.6286_MANUAL.pt"
)

print("Saved successfully!")

Saved successfully!


In [37]:
torch.save({
    "best_f1": 0.6286,
    "best_thresh": 0.54,
    "epoch": 1,
    "model_state_dict": gata_model.state_dict()
},
"./results/GATA_best_F1_0.6286_MANUAL_META.pt"
)

print("Metadata saved!")

Metadata saved!


In [38]:
torch.save(
    gata_model.state_dict(),
    "./results/GATA_best_CURRENT.pt"
)

In [39]:
import torch

meta = torch.load(
    "./results/GATA_best_F1_0.6286_MANUAL_META.pt",
    map_location="cpu"
)

print(meta.keys())
print("F1:", meta["best_f1"])
print("Threshold:", meta["best_thresh"])
print("Epoch:", meta["epoch"])

dict_keys(['best_f1', 'best_thresh', 'epoch', 'model_state_dict'])
F1: 0.6286
Threshold: 0.54
Epoch: 1


In [40]:
import os

print(os.path.abspath("./results/GATA_best_F1_0.6286_MANUAL_META.pt"))

/home/user/Downloads/results/GATA_best_F1_0.6286_MANUAL_META.pt


In [41]:
import shutil

shutil.copy(
    "./results/GATA_best_F1_0.6286_MANUAL_META.pt",
    r"C:\Users\giria\Desktop\GATA_best_F1_0.6286_BACKUP.pt"
)

print("Backup created!")

Backup created!


In [42]:
import os

print(os.path.exists(
    r"C:\Users\giria\Desktop\GATA_best_F1_0.6286_BACKUP.pt"
))

True


In [64]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
from sklearn.metrics import f1_score, precision_score, recall_score
from collections import defaultdict

# -----------------------------
# 1) Load the saved best model
# -----------------------------
CKPT_PATH = "./results/GATA_best_F1_0.6286_MANUAL_META.pt"   # change if needed
ckpt = torch.load(CKPT_PATH, map_location=device)

gata_model.load_state_dict(ckpt["model_state_dict"])
gata_model = gata_model.to(device)
gata_model.eval()

best_thresh = float(ckpt["best_thresh"])
print(f"Loaded checkpoint | best_f1={ckpt['best_f1']:.4f} | thr={best_thresh:.2f} | epoch={ckpt['epoch']}")

# -----------------------------
# 2) Evaluation dataset with PID
# -----------------------------
class EvalRAMDAICDataset(Dataset):
    def __init__(self, cache_dir, pids):
        audio_list = []
        au_list = []
        lm_list = []
        gaze_list = []
        pose_list = []
        labels = []
        genders = []
        pids_all = []

        for pid in pids:
            path = os.path.join(cache_dir, f"{pid}.npy")
            if not os.path.exists(path):
                continue

            data = np.load(path, allow_pickle=True).item()
            n = len(data["label"])

            pids_all.extend([pid] * n)
            audio_list.append(data["audio"])
            au_list.append(data["au"])
            lm_list.append(data["landmark"])
            gaze_list.append(data["gaze"])
            pose_list.append(data["pose"])
            labels.extend(data["label"].tolist())
            genders.extend(data["gender"].tolist())

        self.audio    = torch.FloatTensor(np.concatenate(audio_list))
        self.au       = torch.FloatTensor(np.concatenate(au_list))
        self.landmark = torch.FloatTensor(np.concatenate(lm_list))
        self.gaze     = torch.FloatTensor(np.concatenate(gaze_list))
        self.pose     = torch.FloatTensor(np.concatenate(pose_list))
        self.labels   = torch.tensor(labels, dtype=torch.long)
        self.genders  = torch.tensor(genders, dtype=torch.long)
        self.pids     = np.array(pids_all, dtype=np.int64)

        dep = int(self.labels.sum().item())
        ctrl = len(self.labels) - dep
        print(f"  {len(pids)} participants → {len(self.labels)} windows | dep={dep} ctrl={ctrl}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.audio[idx],
            self.au[idx],
            self.landmark[idx],
            self.gaze[idx],
            self.pose[idx],
            self.labels[idx],
            self.genders[idx],
            self.pids[idx]
        )

# -----------------------------
# 3) Participant-level evaluator
# -----------------------------
def evaluate_participant_level_pid(model, loader, device, threshold=0.5):
    model.eval()
    pid_probs = defaultdict(list)
    pid_labels = {}

    with torch.no_grad():
        for batch in loader:
            audio, au, lm, gaze, pose, label, gender, pid = batch

            audio  = audio.to(device, non_blocking=True)
            au     = au.to(device, non_blocking=True)
            lm     = lm.to(device, non_blocking=True)
            gaze   = gaze.to(device, non_blocking=True)
            pose   = pose.to(device, non_blocking=True)
            gender = gender.to(device, non_blocking=True)

            with autocast():
                logits = model(audio, au, lm, gaze, pose, gender, mc_samples=1)

            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            labels = label.cpu().numpy()
            pids = pid.cpu().numpy()

            for p, pr, y in zip(pids, probs, labels):
                pid_probs[int(p)].append(float(pr))
                pid_labels[int(p)] = int(y)

    final_preds = []
    final_labels = []

    for p in pid_probs:
        mean_prob = float(np.mean(pid_probs[p]))
        pred = 1 if mean_prob >= threshold else 0
        final_preds.append(pred)
        final_labels.append(pid_labels[p])

    f1 = f1_score(final_labels, final_preds, zero_division=0)
    prec = precision_score(final_labels, final_preds, zero_division=0)
    rec = recall_score(final_labels, final_preds, zero_division=0)

    return f1, prec, rec, sum(final_preds), sum(final_labels)

# -----------------------------
# 4) Sanity check on validation
# -----------------------------
val_ds = EvalRAMDAICDataset(CACHE_DIR, val_pids)
val_loader_eval = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

val_f1, val_prec, val_rec, pred_dep, true_dep = evaluate_participant_level_pid(
    gata_model, val_loader_eval, device, threshold=best_thresh
)

print("\n===== VALIDATION CHECK =====")
print(f"F1        : {val_f1:.4f}")
print(f"Precision : {val_prec:.4f}")
print(f"Recall    : {val_rec:.4f}")
print(f"Pred dep  : {pred_dep}")
print(f"True dep  : {true_dep}")
print(f"Threshold : {best_thresh:.2f}")

# -----------------------------
# 5) Final test evaluation
# -----------------------------
test_ds = EvalRAMDAICDataset(CACHE_DIR, test_pids)
test_loader_eval = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_f1, test_prec, test_rec, pred_dep, true_dep = evaluate_participant_level_pid(
    gata_model, test_loader_eval, device, threshold=best_thresh
)

print("\n===== TEST RESULTS =====")
print(f"F1        : {test_f1:.4f}")
print(f"Precision : {test_prec:.4f}")
print(f"Recall    : {test_rec:.4f}")
print(f"Pred dep  : {pred_dep}")
print(f"True dep  : {true_dep}")
print(f"Threshold : {best_thresh:.2f}")

Loaded checkpoint | best_f1=0.6286 | thr=0.54 | epoch=1
  34 participants → 5688 windows | dep=2105 ctrl=3583

===== VALIDATION CHECK =====
F1        : 0.4286
Precision : 0.3529
Recall    : 0.5455
Pred dep  : 17
True dep  : 11
Threshold : 0.54
  44 participants → 7557 windows | dep=2434 ctrl=5123

===== TEST RESULTS =====
F1        : 0.4000
Precision : 0.3333
Recall    : 0.5000
Pred dep  : 21
True dep  : 14
Threshold : 0.54


In [44]:
import os
print(os.path.exists("./results/gata_model.pt"))

True


In [45]:
ckpt_model = torch.load("./results/gata_model.pt", map_location=device)

In [46]:
ckpt["epoch"]

1

In [47]:
print(ckpt["best_f1"])
print(ckpt["best_thresh"])

0.6286
0.54


ABLATION STUDIES


In [23]:
# Run this FIRST before anything else
# Protects your best result from being overwritten

import shutil

shutil.copy(
    "./results/gata_model.pt",
    "./results/GATA_BEST_0.6286_PROTECTED.pt"
)
print("Best model protected ✓")
print("All ablations will save to separate files")

Best model protected ✓
All ablations will save to separate files


In [26]:
baseline_ablation.gata = type('', (), {})()
baseline_ablation.gata.K = 0
baseline_ablation.gata.offset_weights = torch.zeros(
    2, 4, 1, device=device
)

In [28]:
import inspect

src = inspect.getsource(train_gata_final)

for i, line in enumerate(src.splitlines()):
    if ".gata" in line:
        print(i, line)

163                 model.gata.offset_weights, dim=-1)
164             K = model.gata.K


In [29]:
# Cell A1 — Ablation: Synchronized baseline (fixed)

class BaselineModelPatched(BaselineModel):
    """BaselineModel that accepts gender and mc_samples but ignores them."""
    def forward(self, audio, au, landmark, gaze, pose,
                gender, mc_samples=1):
        # Call original BaselineModel forward without gender
        a = self.enc_audio(audio)
        u = self.enc_au(au)
        l = self.enc_landmark(landmark)
        g = self.enc_gaze(gaze)
        p = self.enc_pose(pose)
        a = self.audio_pool(a.permute(0,2,1)).permute(0,2,1)
        fused = a + u + l + g + p
        out   = self.transformer(fused).mean(dim=1)
        return self.classifier(out)


baseline_ablation = BaselineModelPatched()
baseline_ablation.build_encoders(
    audio_dim, au_dim, lm_dim, gaze_dim, pose_dim)
baseline_ablation = baseline_ablation.to(device)
# fake GATA attributes required by trainer
class DummyGATA:
    pass

baseline_ablation.gata = DummyGATA()
baseline_ablation.gata.K = 0
baseline_ablation.gata.offset_weights = torch.zeros(
    (2, 4, 1),
    device=device
)

print("Has gata:", hasattr(baseline_ablation, "gata"))
print("K:", baseline_ablation.gata.K)
print("Shape:", baseline_ablation.gata.offset_weights.shape)

print(f"Baseline params: "
      f"{sum(p.numel() for p in baseline_ablation.parameters()):,}")

baseline_save = "./results/ablation_baseline.pt"

history_baseline, f1_baseline = train_gata_final(
    model      = baseline_ablation,
    train_pids = train_pids,
    val_pids   = val_pids,
    cache_dir  = CACHE_DIR,
    n_epochs   = 80,
    device     = device,
    save_path  = baseline_save,
)

print(f"\nBaseline F1 : {f1_baseline:.4f}")
print(f"GATA-Dep F1 : 0.6286")
print(f"Async contribution: {(0.6286 - f1_baseline)*100:+.2f} pp")

Has gata: True
K: 0
Shape: torch.Size([2, 4, 1])
Baseline params: 2,209,092
Loading train data into RAM...
  107 participants → 16089 windows | dep=4524 ctrl=11565
Loading val data into RAM...
  34 participants → 5688 windows | dep=2105 ctrl=3583
Train → dep:4524 ctrl:11565 ratio:2.56

GATA-Dep FINAL | K=12 | batch=32 | Warmup=10 | Patience=20
Ep   1/80 | Loss 0.7041 | F1 0.5238 | P 0.3548 R 1.0000 | Pred[dep=31 true=11] | Thr=0.52 | Off[M=+0.00s F=+0.00s] | LR 2.0e-05 ← BEST
Ep   2/80 | Loss 0.7006 | F1 0.5294 | P 0.3913 R 0.8182 | Pred[dep=23 true=11] | Thr=0.52 | Off[M=+0.00s F=+0.00s] | LR 3.0e-05 ← BEST
Ep   3/80 | Loss 0.6980 | F1 0.5366 | P 0.3667 R 1.0000 | Pred[dep=30 true=11] | Thr=0.50 | Off[M=+0.00s F=+0.00s] | LR 4.0e-05 ← BEST
Ep   4/80 | Loss 0.6979 | F1 0.6286 | P 0.4583 R 1.0000 | Pred[dep=24 true=11] | Thr=0.50 | Off[M=+0.00s F=+0.00s] | LR 5.0e-05 ← BEST
Ep   5/80 | Loss 0.6970 | F1 0.6154 | P 0.5333 R 0.7273 | Pred[dep=15 true=11] | Thr=0.50 | Off[M=+0.00s F=+0.00s]

In [32]:
batch = next(iter(val_loader))
print(len(batch))

7


In [36]:
val_ds_eval = EvalRAMDAICDataset(
    CACHE_DIR,
    val_pids
)

val_loader_eval = DataLoader(
    val_ds_eval,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

f1, prec, rec, pred_dep, true_dep = \
    evaluate_participant_level_pid(
        baseline_model,
        val_loader_eval,
        device,
        threshold=0.50
)

print("VAL F1 :", f1)
print("VAL P  :", prec)
print("VAL R  :", rec)

  34 participants → 5688 windows | dep=2105 ctrl=3583
VAL F1 : 0.48888888888888893
VAL P  : 0.3235294117647059
VAL R  : 1.0


In [37]:
test_ds_eval = EvalRAMDAICDataset(
    CACHE_DIR,
    test_pids
)

test_loader_eval = DataLoader(
    test_ds_eval,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

f1_test, prec_test, rec_test, pred_dep, true_dep = \
    evaluate_participant_level_pid(
        baseline_model,
        test_loader_eval,
        device,
        threshold=0.50
)

print("TEST F1 :", f1_test)
print("TEST P  :", prec_test)
print("TEST R  :", rec_test)

  44 participants → 7557 windows | dep=2434 ctrl=5123
TEST F1 : 0.4827586206896552
TEST P  : 0.3181818181818182
TEST R  : 1.0


In [30]:
# What this tests:
# Keep async offsets but remove gender conditioning
# Proves gender-aware offset weights add value

# Subclass GATADepModel — override forward to zero gender
class GATADepNoGender(GATADepModel):
    def forward(self, audio, au, landmark, gaze, pose,
                gender, mc_samples=1):
        # Force all participants to use male weights (gender=0)
        gender_zeroed = torch.zeros_like(gender)
        return super().forward(
            audio, au, landmark, gaze, pose,
            gender_zeroed, mc_samples)

nogender_model = GATADepNoGender(K=12)
nogender_model.build_encoders(
    audio_dim, au_dim, lm_dim, gaze_dim, pose_dim)
nogender_model = nogender_model.to(device)

print(f"No-gender model params: "
      f"{sum(p.numel() for p in nogender_model.parameters()):,}")

nogender_save = "./results/ablation_no_gender.pt"

history_nogender, f1_nogender = train_gata_final(
    model      = nogender_model,
    train_pids = train_pids,
    val_pids   = val_pids,
    cache_dir  = CACHE_DIR,
    n_epochs   = 80,
    device     = device,
    save_path  = nogender_save,
)

print(f"\nNo-gender F1     : {f1_nogender:.4f}")
print(f"Full GATA-Dep F1 : 0.6286")
print(f"Gender contribution: {(0.6286 - f1_nogender)*100:+.2f} pp")

No-gender model params: 3,526,412
Loading train data into RAM...
  107 participants → 16089 windows | dep=4524 ctrl=11565
Loading val data into RAM...
  34 participants → 5688 windows | dep=2105 ctrl=3583
Train → dep:4524 ctrl:11565 ratio:2.56

GATA-Dep FINAL | K=12 | batch=32 | Warmup=10 | Patience=20
Ep   1/80 | Loss 0.6927 | F1 0.5641 | P 0.3929 R 1.0000 | Pred[dep=28 true=11] | Thr=0.50 | Off[M=+0.08s F=-0.48s] | LR 2.0e-05 ← BEST
Ep   2/80 | Loss 0.6858 | F1 0.6452 | P 0.5000 R 0.9091 | Pred[dep=20 true=11] | Thr=0.48 | Off[M=+0.08s F=-0.48s] | LR 3.0e-05 ← BEST
Ep   3/80 | Loss 0.6786 | F1 0.5714 | P 0.4706 R 0.7273 | Pred[dep=17 true=11] | Thr=0.50 | Off[M=+0.08s F=-0.48s] | LR 4.0e-05
Ep   4/80 | Loss 0.6591 | F1 0.4348 | P 0.4167 R 0.4545 | Pred[dep=12 true=11] | Thr=0.50 | Off[M=+0.04s F=-0.48s] | LR 5.0e-05
Ep   5/80 | Loss 0.6327 | F1 0.6207 | P 0.5000 R 0.8182 | Pred[dep=18 true=11] | Thr=0.46 | Off[M=+0.04s F=-0.48s] | LR 6.0e-05
Ep   6/80 | Loss 0.6057 | F1 0.6000 | P 0.

KeyboardInterrupt: 

In [20]:
# What this tests:
# How much does temporal offset range matter?
# K=0 = no offset (effectively synchronized)
# K=4 = ±0.16s, K=8 = ±0.32s, K=12 = ±0.48s (your full model)

k_results = {'K=12 (full)': 0.6286}  # your existing result

for K_val in [8]:
    print(f"\n{'='*55}")
    print(f"Training K={K_val}")
    print(f"{'='*55}")

    model_k = GATADepModel(K=K_val)
    model_k.build_encoders(
        audio_dim, au_dim, lm_dim, gaze_dim, pose_dim)
    model_k = model_k.to(device)

    save_k = f"./results/ablation_K{K_val}.pt"

    history_k, f1_k = train_gata_final(
        model      = model_k,
        train_pids = train_pids,
        val_pids   = val_pids,
        cache_dir  = CACHE_DIR,
        n_epochs   = 80,
        device     = device,
        save_path  = save_k,
    )

    k_results[f'K={K_val}'] = f1_k
    print(f"K={K_val} F1: {f1_k:.4f}")

print(f"\nK ablation summary:")
print(f"{'Variant':<15} {'F1':>8}")
print("-"*25)
for k, f in sorted(k_results.items()):
    marker = " ← yours" if k == "K=12 (full)" else ""
    print(f"{k:<15} {f:>8.4f}{marker}")


Training K=8
Loading train data into RAM...
  107 participants → 16089 windows | dep=4524 ctrl=11565
Loading val data into RAM...
  34 participants → 5688 windows | dep=2105 ctrl=3583
Train → dep:4524 ctrl:11565 ratio:2.56

GATA-Dep FINAL | K=12 | batch=32 | Warmup=10 | Patience=20
Ep   1/80 | Loss 0.6699 | F1 0.6286 | P 0.4583 R 1.0000 | Pred[dep=24 true=11] | Thr=0.48 | Off[M=+0.32s F=+0.20s] | LR 2.0e-05 ← BEST
Ep   2/80 | Loss 0.5743 | F1 0.5789 | P 0.4074 R 1.0000 | Pred[dep=27 true=11] | Thr=0.46 | Off[M=+0.32s F=+0.00s] | LR 3.0e-05
Ep   3/80 | Loss 0.4782 | F1 0.5556 | P 0.4000 R 0.9091 | Pred[dep=25 true=11] | Thr=0.38 | Off[M=-0.20s F=+0.00s] | LR 4.0e-05
Ep   4/80 | Loss 0.4094 | F1 0.5556 | P 0.4000 R 0.9091 | Pred[dep=25 true=11] | Thr=0.38 | Off[M=-0.04s F=+0.00s] | LR 5.0e-05
Ep   5/80 | Loss 0.3472 | F1 0.5405 | P 0.3846 R 0.9091 | Pred[dep=26 true=11] | Thr=0.12 | Off[M=-0.04s F=+0.00s] | LR 6.0e-05
Ep   6/80 | Loss 0.3164 | F1 0.5556 | P 0.4000 R 0.9091 | Pred[dep=25

KeyboardInterrupt: 

In [24]:
import torch
import torch.nn as nn

class MissingModalityWrapper(nn.Module):

    def __init__(self, model, modality):
        super().__init__()
        self.model = model
        self.modality = modality

    def forward(
        self,
        audio,
        au,
        landmark,
        gaze,
        pose,
        gender,
        mc_samples=1
    ):

        if self.modality == "audio":
            audio = torch.zeros_like(audio)

        elif self.modality == "au":
            au = torch.zeros_like(au)

        elif self.modality == "landmark":
            landmark = torch.zeros_like(landmark)

        elif self.modality == "gaze":
            gaze = torch.zeros_like(gaze)

        elif self.modality == "pose":
            pose = torch.zeros_like(pose)

        return self.model(
            audio,
            au,
            landmark,
            gaze,
            pose,
            gender,
            mc_samples
        )

In [25]:
model.eval()

full_f1, full_prec, full_rec, _, _ = \
    evaluate_participant_level_pid(
        model,
        val_loader_eval,
        device,
        threshold=best_thresh
    )

print("Full Model")
print("F1 =", full_f1)
print("Precision =", full_prec)
print("Recall =", full_rec)

NameError: name 'model' is not defined

In [26]:
for name in globals():
    if "model" in name.lower():
        print(name)

RuntimeError: dictionary changed size during iteration

In [27]:
for name in globals():
    if "gata" in name.lower():
        print(name)

GATAFusion
GATADepModel
gata_model
train_gata_final
gata_save


In [28]:
gata_model.eval()

print("Model loaded successfully")

Model loaded successfully


In [31]:
for name in globals():
    if "eval" in name.lower():
        print(name)

evaluate_participant_level


In [32]:
import inspect
print(inspect.signature(evaluate_participant_level))

(model, loader, device, threshold=0.5)


In [33]:
import torch
import torch.nn as nn

class MissingModalityWrapper(nn.Module):

    def __init__(self, model, modality):
        super().__init__()
        self.model = model
        self.modality = modality

    def forward(
        self,
        audio,
        au,
        landmark,
        gaze,
        pose,
        gender
    ):

        if self.modality == "audio":
            audio = torch.zeros_like(audio)

        elif self.modality == "au":
            au = torch.zeros_like(au)

        elif self.modality == "landmark":
            landmark = torch.zeros_like(landmark)

        elif self.modality == "gaze":
            gaze = torch.zeros_like(gaze)

        elif self.modality == "pose":
            pose = torch.zeros_like(pose)

        return self.model(
            audio,
            au,
            landmark,
            gaze,
            pose,
            gender
        )

In [39]:
for name in globals():
    if "loader" in name.lower():
        print(name)

__loader__
DataLoader
val_loader
test_loader


In [40]:
print(evaluate_participant_level.__code__.co_firstlineno)

60


In [42]:
import torch
import torch.nn as nn

# =====================================================
# Missing Modality Wrapper
# =====================================================

class MissingModalityWrapper(nn.Module):
    def __init__(self, model, modality):
        super().__init__()
        self.model = model
        self.modality = modality

    def forward(self, audio, au, lm, gaze, pose, gender, mc_samples=1):

        if self.modality == "audio":
            audio = torch.zeros_like(audio)

        elif self.modality == "au":
            au = torch.zeros_like(au)

        elif self.modality == "landmark":
            lm = torch.zeros_like(lm)

        elif self.modality == "gaze":
            gaze = torch.zeros_like(gaze)

        elif self.modality == "pose":
            pose = torch.zeros_like(pose)

        return self.model(
            audio,
            au,
            lm,
            gaze,
            pose,
            gender,
            mc_samples=mc_samples
        )

In [47]:
class MissingModalityWrapper(nn.Module):
    def __init__(self, model, modality):
        super().__init__()
        self.model = model
        self.modality = modality

    def forward(
        self,
        audio,
        au,
        landmark,
        gaze,
        pose,
        gender,
        mc_samples=1
    ):

        if self.modality == "audio":
            audio = torch.zeros_like(audio)

        elif self.modality == "au":
            au = torch.zeros_like(au)

        elif self.modality == "landmark":
            landmark = torch.zeros_like(landmark)

        elif self.modality == "gaze":
            gaze = torch.zeros_like(gaze)

        elif self.modality == "pose":
            pose = torch.zeros_like(pose)

        return self.model(
            audio,
            au,
            landmark,
            gaze,
            pose,
            gender,
            mc_samples=mc_samples
        )

In [59]:
import torch
import torch.nn as nn
from sklearn.metrics import f1_score

class MissingModalityWrapper(nn.Module):
    def __init__(self, model, modality):
        super().__init__()
        self.model = model
        self.modality = modality

    def forward(self, audio, au, landmark, gaze, pose, gender, mc_samples=1):

        if self.modality == "audio":
            audio = torch.zeros_like(audio)

        elif self.modality == "au":
            au = torch.zeros_like(au)

        elif self.modality == "landmark":
            landmark = torch.zeros_like(landmark)

        elif self.modality == "gaze":
            gaze = torch.zeros_like(gaze)

        elif self.modality == "pose":
            pose = torch.zeros_like(pose)

        return self.model(
            audio, au, landmark, gaze, pose,
            gender, mc_samples=mc_samples
        )


def eval_window_f1(model, loader, threshold=0.5):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:

            audio, au, landmark, gaze, pose, label, gender = batch

            logits = model(
                audio.to(device),
                au.to(device),
                landmark.to(device),
                gaze.to(device),
                pose.to(device),
                gender.to(device),
                mc_samples=1
            )

            probs = torch.softmax(logits, dim=1)[:,1]

            preds = (probs >= threshold).long()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(label.numpy())

    return f1_score(all_labels, all_preds)

In [60]:
modalities = [
    ("audio","Without Audio"),
    ("au","Without AU"),
    ("landmark","Without Landmark"),
    ("gaze","Without Gaze"),
    ("pose","Without Pose")
]

full_f1 = eval_window_f1(
    gata_model,
    val_loader,
    threshold=0.5
)

print("Full =", full_f1)

for mod_key, mod_name in modalities:

    wrapped = MissingModalityWrapper(
        gata_model,
        mod_key
    ).to(device)

    f1 = eval_window_f1(
        wrapped,
        val_loader,
        threshold=0.5
    )

    print(
        f"{mod_name:20s} "
        f"F1={f1:.3f} "
        f"Drop={full_f1-f1:.3f}"
    )

Full = 0.2963562753036438
Without Audio        F1=0.410 Drop=-0.114
Without AU           F1=0.250 Drop=0.047
Without Landmark     F1=0.305 Drop=-0.009
Without Gaze         F1=0.334 Drop=-0.037
Without Pose         F1=0.184 Drop=0.112


In [61]:
class MissingModalityWrapper(nn.Module):
    def __init__(self, model, modality):
        super().__init__()
        self.model = model
        self.modality = modality

    def forward(
        self,
        audio,
        au,
        landmark,
        gaze,
        pose,
        gender,
        mc_samples=1
    ):

        if self.modality == "audio":
            audio = torch.zeros_like(audio)

        elif self.modality == "au":
            au = torch.zeros_like(au)

        elif self.modality == "landmark":
            landmark = torch.zeros_like(landmark)

        elif self.modality == "gaze":
            gaze = torch.zeros_like(gaze)

        elif self.modality == "pose":
            pose = torch.zeros_like(pose)

        return self.model(
            audio,
            au,
            landmark,
            gaze,
            pose,
            gender,
            mc_samples=mc_samples
        )

In [65]:
modalities = [
    ("audio", "Without Audio"),
    ("au", "Without AU"),
    ("landmark", "Without Landmark"),
    ("gaze", "Without Gaze"),
    ("pose", "Without Pose")
]

# Full model first
full_f1, full_prec, full_rec, _, _ = evaluate_participant_level_pid(
    gata_model,
    val_loader_eval,
    device,
    threshold=best_thresh
)

print("\nFULL MODEL")
print(f"F1 = {full_f1:.4f}")

results = []

for mod_key, mod_name in modalities:

    wrapped_model = MissingModalityWrapper(
        gata_model,
        mod_key
    ).to(device)

    f1, prec, rec, _, _ = evaluate_participant_level_pid(
        wrapped_model,
        val_loader_eval,
        device,
        threshold=best_thresh
    )

    drop = full_f1 - f1

    results.append([
        mod_name,
        round(f1, 3),
        round(drop, 3)
    ])

    print(f"{mod_name:20s}  F1={f1:.4f}  Drop={drop:.4f}")


FULL MODEL
F1 = 0.4286
Without Audio         F1=0.6087  Drop=-0.1801
Without AU            F1=0.2105  Drop=0.2180
Without Landmark      F1=0.5556  Drop=-0.1270
Without Gaze          F1=0.5500  Drop=-0.1214
Without Pose          F1=0.2500  Drop=0.1786


In [66]:
full_f1, full_prec, full_rec, _, _ = evaluate_participant_level_pid(
    gata_model,
    val_loader_eval,
    device,
    threshold=best_thresh
)

print(full_f1)

0.42857142857142855
